In [ ]:
import os
from collections.abc import MutableMapping
from copy import deepcopy
from typing import Any, cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.signal import windows

from tabletop_py.gaze.preprocess import (
    calculate_eyelink_speed,
    calculate_eyelink_speed_savgol,
    clean_eyelink_data,
    preprocess_data,
    reindex_and_interpolate,
    smooth_eyelink_data,
)

pd.options.mode.copy_on_write = True

In [ ]:
session_dir = (
    "/tabletop/bags_i_care_about/session_2026-02-18_20-16-32_training"
)
config_path = os.path.join(
    os.environ["TABLETOP_DIR"], "config", "gaze_estimation.yaml"
)
with open(config_path, "r") as f:
    orig_config = yaml.safe_load(f)

config = deepcopy(orig_config)
eyelink_config = cast(
    MutableMapping[str, Any], config["preprocess"]["eyelink"]
)

path = os.path.join(session_dir, "raw_eyelink.csv")


def foo():
    df = pd.read_csv(path, index_col=False)

    df = df[(df["time"] < 20) | (df["time"] > 21)]

    df = clean_eyelink_data(df, **eyelink_config["clean"])

    steady_idx = np.arange(
        df["time"].min(), df["time"].max(), 1 / config["eyelink_freq"]
    )

    df = reindex_and_interpolate(
        df,
        steady_idx,
        on="time",
        **eyelink_config["reindex_and_interpolate"],
    )
    return df

In [ ]:
min_speed = 100
marker_size = 0.5
start_time = 32
end_time = 33
start_time = 0
end_time = float("inf")
window = 0.02
polyorder = 2
gaussian_std = 2.0

methods = [
    "savgol_speed",
    "savgol_smooth",
    "rolling_smooth",
    "gaussian_smooth",
]
dfs = {}
for method in methods:
    df = foo()

    match method:
        case "savgol_speed":
            df[["left_speed", "right_speed"]] = calculate_eyelink_speed_savgol(
                df,
                freq=config["eyelink_freq"],
                window=window,
                polyorder=polyorder,
            )
        case "savgol_smooth":
            df = smooth_eyelink_data(
                df,
                freq=config["eyelink_freq"],
                method="savgol",
                window=window,
                polyorder=polyorder,
            )
            df[["left_speed", "right_speed"]] = calculate_eyelink_speed(df)
        case "rolling_smooth":
            df = smooth_eyelink_data(
                df,
                freq=config["eyelink_freq"],
                method="rolling",
                window=window,
            )
            df[["left_speed", "right_speed"]] = calculate_eyelink_speed(df)
        case "gaussian_smooth":
            df = smooth_eyelink_data(
                df,
                freq=config["eyelink_freq"],
                method="rolling",
                window=window,
                win_type="gaussian",
                win_kwargs={"std": gaussian_std},
            )
            df[["left_speed", "right_speed"]] = calculate_eyelink_speed(df)

    invalid_mask = (df["time"] < start_time) | (df["time"] > end_time)
    df = df[~invalid_mask]

    dfs[method] = df

fig, ax = plt.subplots(
    len(methods),
    1,
    sharex=True,
    sharey=True,
    figsize=(10, 10),
    tight_layout=True,
)
fig.suptitle("Position")
for i, (method, df) in enumerate(dfs.items()):
    valid_df = df[df["left_speed"] >= min_speed]
    invalid_df = df[df["left_speed"] < min_speed]
    # print(f"Num samples too slow: {invalid_df.shape[0]}")

    ax[i].set_title(
        f"Method: {method} | Too slow count: {invalid_df.shape[0]}"
    )
    ax[i].scatter(valid_df["time"], valid_df["left_y"], s=marker_size)
    ax[i].scatter(invalid_df["time"], invalid_df["left_y"], s=2 * marker_size)
plt.show()

fig, ax = plt.subplots(
    len(methods),
    1,
    sharex=True,
    sharey=True,
    figsize=(10, 10),
    tight_layout=True,
)
fig.suptitle("Speed")
for i, (method, df) in enumerate(dfs.items()):
    valid_df = df[df["left_speed"] >= min_speed]
    invalid_df = df[df["left_speed"] < min_speed]

    ax[i].set_title(
        f"Method: {method} | Too slow count: {invalid_df.shape[0]}"
    )
    ax[i].scatter(valid_df["time"], valid_df["left_speed"], s=marker_size)
    ax[i].scatter(
        invalid_df["time"], invalid_df["left_speed"], s=2 * marker_size
    )
plt.show()

fig, ax = plt.subplots(
    len(methods),
    1,
    sharex=True,
    sharey=True,
    figsize=(10, 10),
    tight_layout=True,
)
fig.suptitle("Speed Histogram")
for i, (method, df) in enumerate(dfs.items()):
    ax[i].set_title(f"Method: {method}")
    ax[i].hist(df["left_speed"], bins=100)
plt.show()

M = int(window * config["eyelink_freq"])
x = np.arange(M)
y = windows.gaussian(M, gaussian_std)
plt.plot(x, y)
plt.show()

In [ ]:
# min_speed = 200
# marker_size = 0.5
# # start_time = 20
# # end_time = float("inf")
# start_time = 32
# end_time = 36
# window = 0.05
# gaussian_std = 2.0
#
smooth_configs = []
smooth_configs.append(
    {
        "method": "savgol",
        "window": window,
        "polyorder": 3,
    }
)
smooth_configs.append(
    {
        "method": "rolling",
        "window": window,
    }
)
smooth_configs.append(
    {
        "method": "rolling",
        "window": window,
        "win_type": "gaussian",
        "win_kwargs": {"std": gaussian_std},
    }
)
for smooth_config in smooth_configs:
    print(f"Smoothing config: {smooth_config}")

    config["preprocess"]["eyelink"]["smooth"] = smooth_config

    df, _, _ = preprocess_data(
        session_dir,
        config,
        marker_idx=0,
        start_time=start_time,
        end_time=end_time,
    )
    df[["left_speed", "right_speed"]] = calculate_eyelink_speed(df)

    # df = df[df["left_speed"] < 2000]

    valid_df = df[df["left_speed"] >= min_speed]
    invalid_df = df[df["left_speed"] < min_speed]
    print(f"Num samples too slow: {invalid_df.shape[0]}")

    fig, ax = plt.subplots(2, 1, sharex=True)
    ax[0].scatter(valid_df["time"], valid_df["left_speed"], s=marker_size)
    ax[1].scatter(valid_df["time"], valid_df["left_x"], s=marker_size)
    ax[0].scatter(invalid_df["time"], invalid_df["left_speed"], s=marker_size)
    ax[1].scatter(invalid_df["time"], invalid_df["left_x"], s=marker_size)
    plt.show()
    df["left_speed"].hist(bins=50)
    plt.show()

M = int(window * config["eyelink_freq"])
x = np.arange(M)
y = windows.gaussian(M, gaussian_std)
plt.plot(x, y)
plt.show()